In [ ]:
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, Dict, Any, List

In [ ]:
class PetProfile:
    """
    Pet data
    """
    pet_id: str
    breed: str # for searching any special attribute of the breed
    birth_date: datetime
    is_high_energy: bool  # e.g., Vizsla 
    postcode: str         # For seasonal/geographic tick risk 

@dataclass
class CheckInData:
    """Pet check-in data, recorded by owner"""
    timestamp: datetime
    appetite: int          # 1-5 scale
    energy: int            # 1-5 scale
    sleep_quality: int     # 1-5 scale
    water_intake: str      # "Normal", "Increased" 
    toileting_blood: bool  # Acute override trigger 
    activity_minutes: int
    weight: int # Update once a month
    owner_struggling: bool # N/A for score, but flags dashboard 
    

In [ ]:
class VetInteractionLog:
    """List of yes and no question and will be input in this to account to Vitality score"""

In [ ]:
class PetDiseaseIndicator:
    """
    Evaluates check-in data specifically for clinical red flags and maps them to potential disease categories for vet reference.
    """
    def __init__(self, pet: PetProfile):
        self.pet = pet

    def evaluate(self, current: CheckInData) -> List[Dict[str, str]]:
        clinical_flags = []

        # Indicator 1: Increased Water Intake
        if current.water_intake == "Increased":
            clinical_flags.append({
                "trigger": "Increased Water Intake",
                "severity": "High",
                "potential_diseases": ["Diabetes", "Kidney Disease", "Cushing's Disease"],
                "action": "Requires bloodwork and urinalysis."
            })

        # Indicator 2: Blood in Stool
        if current.toileting_blood:
            clinical_flags.append({
                "trigger": "Blood in Stool",
                "severity": "Acute",
                "potential_diseases": ["Gastroenteritis", "Parasitic Infection", "Colitis", "Toxicity"],
                "action": "Immediate veterinary attention recommended."
            })

        # Indicator 3: Confirmed Tick
        if current.tick_found:
            clinical_flags.append({
                "trigger": "Tick Found",
                "severity": "Acute",
                "potential_diseases": ["Tick Paralysis", "Tick-borne illness (e.g., Ehrlichiosis)"],
                "action": "Immediate search for additional ticks and symptom monitoring; vet visit if symptomatic."
            })

        # Indicator 4: Sudden Energy/Appetite Collapse ("Moment 1" cluster)
        if current.energy <= 1 and current.appetite <= 1:
            clinical_flags.append({
                "trigger": "Lethargy & Anorexia Cluster",
                "severity": "Acute",
                "potential_diseases": ["Systemic Infection", "Severe Pain", "Obstruction"],
                "action": "Immediate veterinary attention recommended."
            })

        return clinical_flags

In [ ]:
class VitalityScoreEngine:
    """
    Vitality score calculator
    """
    def __init__(self, pet: PetProfile, history: list):
        self.pet = pet
        self.history = history # Previous 4 weeks for baseline

    def calculate_score(self, current: CheckInData) -> Dict[str, Any]:
        # Check for Acute Overrides
        if current.toileting_blood or current.water_intake == "Increased":
            return {"score": 40, "zone": "Action Needed", "override": True}

        # Apply Breed-Based Weighting (set low, medium, high weight) 
        weights = {"activity": 0.2, "appetite": 0.25, "energy": 0.25, "sleep": 0.3}
        if self.pet.is_high_energy:
            weights["activity"] = 0.4 # Vizsla adjustment 
            weights["sleep"] = 0.1

        # Detect Signals (Combined symptoms) 
        raw_score = 100
        if current.energy < 3 and current.appetite < 3:
            raw_score -= 30  # "Moment 1" signal cluster 
        
        # Handle Baseline Period 
        if len(self.history) < 4:
            return {"score": None, "status": "Building your baseline"}

        return {"score": max(raw_score, 0), "zone": self._get_zone(raw_score)}

    def _get_zone(self, score):
        if score >= 85: return "Bright Green" # Healthy
        if score >= 70: return "Medium Green" # Standard
        if score >= 50: return "Yellow" # Cautious
        return "Red" # Action Needed